# SentiVox — Multilingual Sentiment Analysis Model Training
Fine-tunes google/muril-base-cased to score text on four sentiment axes —
**positive**, **negative**, **neutral** and **ambiguous**.

Training data format (each score is between 0 and 1, and the four scores of a sample always sum to 1):
```
[
    {
        "text": "My Text",
        "senti": { "pos": 0.7, "neg": 0.2, "neu": 0.1, "amb": 0.0 }
    },
    ...
]
```

Because the four scores form a distribution, the model has one output per sentiment followed by a
**softmax**, and is trained with soft-label cross-entropy against the `senti` scores.
A text can carry several sentiments at once (e.g. positive 0.5, negative 0.4, neutral 0.05,
ambiguous 0.05), so the model always reports all four scores. The dominant sentiment is the one
with the highest score.

Output format:
```
{
    "sentiment": "positive" | "negative" | "neutral" | "ambiguous",
    "scores": { "positive": <0-1>, "negative": <0-1>, "neutral": <0-1>, "ambiguous": <0-1> }
}
```
The four scores in `"scores"` sum to 1.

Designed for Google Colab (free-tier GPU).
================================================================================


# 0. Install Dependencies (run once on Colab)


In [1]:
from os import system
from json import load as jsonLoad, dump as jsonDump
from typing import Dict, List

URL_OF_DATASETS = [
    "https://raw.githubusercontent.com/saksham-joshi/Senti-Vox-CB-Model/refs/heads/main/dataset/cleaned/go-sentiment-normalized.json",
    "https://raw.githubusercontent.com/saksham-joshi/Senti-Vox-CB-Model/refs/heads/main/dataset/cleaned/sst5-normalized.json"
]

COMBINED_DATASET : List[Dict[str, str | Dict[str, float]]] = []

for url in URL_OF_DATASETS :
    system("wget -O temp.json " + url)
    with open("temp.json", "r", encoding="utf-8") as file :
        COMBINED_DATASET.extend(list(jsonLoad(file)))
    system("rm temp.json")

DATASET_NAME = "combined-dataset.json"

with open(DATASET_NAME, "w", encoding="utf-8") as file :
    jsonDump(COMBINED_DATASET, file, ensure_ascii=False, allow_nan=False, indent=None, separators=(",", ":"))

In [2]:
URL_OF_TOKENIZER = "https://raw.githubusercontent.com/saksham-joshi/Senti-Vox-CB-Model/refs/heads/main/scripts/tokenizer.py"
URL_OF_VALUES = "https://raw.githubusercontent.com/saksham-joshi/Senti-Vox-CB-Model/refs/heads/main/scripts/values.json"

! mkdir scripts

! wget -O scripts/tokenizer.py $URL_OF_TOKENIZER
! wget -O scripts/values.json $URL_OF_VALUES

! pip install transformers datasets accelerate scikit-learn torch numpy pandas matplotlib seaborn huggingface_hub

! ls

--2026-10-29 06:18:27--  https://raw.githubusercontent.com/saksham-joshi/Senti-Vox-CB-Model/refs/heads/main/scripts/tokenizer.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 46327 (45K) [text/plain]
Saving to: ‘scripts/tokenizer.py’

scripts/tokenizer.p 100%[===================>]  45.24K  --.-KB/s    in 0.007s  

2026-10-29 06:18:27 (6.13 MB/s) - ‘scripts/tokenizer.py’ saved [46327/46327]

--2026-10-29 06:18:27--  https://raw.githubusercontent.com/saksham-joshi/Senti-Vox-CB-Model/refs/heads/main/scripts/values.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, 


# 1. Imports


In [3]:
import os
import json
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
)
import matplotlib
matplotlib.use("Agg")  # Non-interactive backend (safe for Colab/headless)
import matplotlib.pyplot as plt
import seaborn as sns



# 2. Configuration


In [4]:
class Config:
    """Central configuration for the training pipeline."""
    # Model
    MODEL_NAME = "google/muril-base-cased"
    MAX_LENGTH = 128               # Token limit per input
    NUM_LABELS = 4                 # positive, negative, neutral, ambiguous

    # Dataset scores are 0-1 and the four scores of a sample sum to 1
    SCORE_SUM_TOLERANCE = 1e-3     # Samples whose scores sum further from 1 are dropped

    # Training hyperparameters (tuned for Colab free-tier T4 GPU)
    BATCH_SIZE = 32
    GRADIENT_ACCUMULATION_STEPS = 2  # Effective batch size = 64
    LEARNING_RATE = 2e-5
    WEIGHT_DECAY = 0.01
    NUM_EPOCHS = 7
    WARMUP_RATIO = 0.1
    FP16 = True                    # Mixed precision for faster training

    # Early stopping
    EARLY_STOPPING_PATIENCE = 2
    METRIC_FOR_BEST_MODEL = "mae"  # Mean absolute error of the 0-1 scores
    GREATER_IS_BETTER = False      # Lower MAE is better

    # Data split
    SPLIT_SEED = 42                # Shuffle seed before the train/val/test split

    # Paths
    OUTPUT_DIR = "./models/sentivox-muril-sentiment"
    LOGGING_DIR = "./models/logs"
    RESULTS_DIR = "./models/results"

config = Config()

# Create output directories
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.LOGGING_DIR, exist_ok=True)
os.makedirs(config.RESULTS_DIR, exist_ok=True)

# LABELS IN DATASET

In [5]:
# Keys inside each sample's "senti" dict, and the human-readable label for each.
# The order here is the order of the model's output vector.
SENTI_KEYS = ["pos", "neg", "neu", "amb"]
SENTIMENT_LABELS = ["positive", "negative", "neutral", "ambiguous"]


# 3. Load, Validate & Split Data


In [6]:
TRAINING_DATA = []
VALIDATION_DATA = []
TESTING_DATA = []

def is_valid_sample(item) -> bool:
    """
    A sample needs non-empty text and a numeric 0-1 score for every sentiment key,
    and its four scores must sum to 1.
    """
    if not isinstance(item, dict) or not str(item.get("text", "")).strip():
        return False
    senti = item.get("senti")
    if not isinstance(senti, dict):
        return False
    scores = [senti.get(key) for key in SENTI_KEYS]
    if not all(isinstance(score, (int, float)) and 0 <= score <= 1 for score in scores):
        return False
    return abs(sum(scores) - 1) <= config.SCORE_SUM_TOLERANCE

with open(DATASET_NAME, "r", encoding="utf-8") as file:
    print("Successfully loaded Dataset File")
    lst = json.load(file)
    print("Successfully Parsed Dataset")

    """
        FORMAT OF DATA:
        {
            "text": "",
            "senti": { "pos": 0-1, "neg": 0-1, "neu": 0-1, "amb": 0-1 }
        }
        The four scores of a sample sum to 1.
    """
    valid = [item for item in lst if is_valid_sample(item)]
    print(f"Dropped {len(lst) - len(valid)} malformed samples")
    lst = valid

    # The dataset is merged from several sources, so shuffle before splitting
    random.Random(config.SPLIT_SEED).shuffle(lst)

    training_data_size = int(len(lst) * 0.6)
    validation_data_size = int(len(lst) * 0.8)

    TRAINING_DATA.extend(lst[:training_data_size])
    print("Succesfully created sample of 'TRAINING DATA'")
    VALIDATION_DATA.extend(lst[training_data_size:validation_data_size])
    print("Succesfully created sample of 'VALIDATION DATA'")
    TESTING_DATA.extend(lst[validation_data_size:])
    print("Succesfully created sample of 'TESTING DATA'")

print("\nSizes of Samples:")
print("      Training Data: ", len(TRAINING_DATA))
print("      Validation Data: ", len(VALIDATION_DATA))
print("      Testing Data: ", len(TESTING_DATA))

print("\n Glimpse of Data:")
print("      Training Data: ", TRAINING_DATA[:2])
print("      Validation Data: ", VALIDATION_DATA[:2])
print("      Testing Data: ", TESTING_DATA[:2])


Successfully loaded Dataset File
Successfully Parsed Dataset
Dropped 0 malformed samples
Succesfully created sample of 'TRAINING DATA'
Succesfully created sample of 'VALIDATION DATA'
Succesfully created sample of 'TESTING DATA'

Sizes of Samples:
      Training Data:  129814
      Validation Data:  43272
      Testing Data:  43272

 Glimpse of Data:
      Training Data:  [{'text': 'if there is an ordered psychological work - up , will it be 3 days and 3 nights and will they decide if she needs to stay longer ? thank you', 'senti': {'pos': 1.0, 'neg': 0.0, 'neu': 0.0, 'amb': 0.0}}, {'text': 'underrated joke ... haha', 'senti': {'pos': 1.0, 'neg': 0.0, 'neu': 0.0, 'amb': 0.0}}]
      Validation Data:  [{'text': 'way too tired for work today so taking the day off . feels pretty great !', 'senti': {'pos': 1.0, 'neg': 0.0, 'neu': 0.0, 'amb': 0.0}}, {'text': 'broke - ass street smarts are the real street smarts .', 'senti': {'pos': 1.0, 'neg': 0.0, 'neu': 0.0, 'amb': 0.0}}]
      Testing Dat


# 4. Score → Sentiment Helpers

- ### Converts model logits into 0-1 scores (summing to 1) and picks the dominant sentiment.

In [7]:
def softmax(x):
    x = np.asarray(x, dtype=np.float64)
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def logits_to_scores(logits) -> np.ndarray:
    """Model logits → sentiment scores in [0, 1]; each row sums to 1."""
    return softmax(logits)


def dominant_indices(score_matrix) -> np.ndarray:
    """
    Index of the dominant sentiment for each row of an (N, 4) score matrix:
    the sentiment with the highest score (ties go to the earlier label in SENTIMENT_LABELS).
    """
    return np.atleast_2d(np.asarray(score_matrix, dtype=np.float64)).argmax(axis=1)


def scores_to_sentiment(scores) -> str:
    """Dominant sentiment label for a single 4-element score vector."""
    return SENTIMENT_LABELS[int(dominant_indices(scores)[0])]


# 5. PyTorch Dataset


In [8]:
print("\n[Step 5] Initializing tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)

class SentimentScoreDataset(Dataset):
    """
    Sentiment-score dataset.

    Each sample is tokenized and its "senti" scores (0-1) are converted to a
    float target vector of length 4 (ordered as SENTI_KEYS). The vector is
    re-normalized to sum to exactly 1, since it is the target distribution
    for the soft-label cross-entropy loss.
    """

    def __init__(self, data: list[dict], tokenizer, max_length: int):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item["text"]
        senti = item["senti"]

        # Tokenize
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        # Soft target distribution over the four sentiments
        labels = torch.tensor([senti[key] for key in SENTI_KEYS], dtype=torch.float)
        labels = labels / labels.sum()

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": labels,
        }


print("[Step 5] Building datasets...")
train_dataset = SentimentScoreDataset(TRAINING_DATA, tokenizer, config.MAX_LENGTH)
val_dataset = SentimentScoreDataset(VALIDATION_DATA, tokenizer, config.MAX_LENGTH)
test_dataset = SentimentScoreDataset(TESTING_DATA, tokenizer, config.MAX_LENGTH)
print(f"  Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")


[Step 5] Initializing tokenizer...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

[Step 5] Building datasets...
  Train: 129814 | Val: 43272 | Test: 43272



# 6. Model Initialization


In [9]:
print("\n[Step 6] Loading MuRIL model for sentiment-score prediction...")
# The four scores of a sample sum to 1, so they are treated as a probability distribution:
# softmax over the four logits gives the predicted scores, and SoftLabelTrainer (step 8)
# trains them with cross-entropy against the "senti" scores. A mixed text still gets a
# share on every sentiment it carries (e.g. positive 0.5, negative 0.4).
# "single_label_classification" records in config.json that the outputs are softmax-normalized.
model = AutoModelForSequenceClassification.from_pretrained(
    config.MODEL_NAME,
    num_labels=config.NUM_LABELS,
    problem_type="single_label_classification",
    id2label={i: label for i, label in enumerate(SENTIMENT_LABELS)},
    label2id={label: i for i, label in enumerate(SENTIMENT_LABELS)},
)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Using device: {device}")
model.to(device)

# Print model parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")


[Step 6] Loading MuRIL model for sentiment-score prediction...


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  953MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

  Using device: cuda


model.safetensors: reconstructing file:   0%|          |  0.00B /  953MB            

model.safetensors: downloading bytes:           |  0.00B            

  Total parameters:     237,559,300
  Trainable parameters: 237,559,300


In [10]:
from scripts.tokenizer import register_placeholders

register_placeholders(tokenizer, model)

9


# 7. Metrics Computation


In [11]:
def compute_metrics(eval_pred):
    """
    Compute sentiment-score metrics for the Trainer.

    Score metrics (on the 0-1 scale):
      - mae / rmse over all four sentiments, plus mae_<key> per sentiment.
    Dominant-sentiment metrics (highest score vs. highest true score):
      - accuracy, f1_macro, precision_macro, recall_macro.
    """
    logits, labels = eval_pred
    pred_scores = logits_to_scores(logits)
    true_scores = np.asarray(labels, dtype=np.float64)

    metrics = {
        "mae": mean_absolute_error(true_scores, pred_scores),
        "rmse": float(np.sqrt(mean_squared_error(true_scores, pred_scores))),
    }
    for i, key in enumerate(SENTI_KEYS):
        metrics[f"mae_{key}"] = mean_absolute_error(true_scores[:, i], pred_scores[:, i])

    true_dom = dominant_indices(true_scores)
    pred_dom = dominant_indices(pred_scores)
    label_ids = list(range(len(SENTIMENT_LABELS)))

    metrics["accuracy"] = accuracy_score(true_dom, pred_dom)
    metrics["f1_macro"] = f1_score(true_dom, pred_dom, labels=label_ids, average="macro", zero_division=0)
    metrics["precision_macro"] = precision_score(true_dom, pred_dom, labels=label_ids, average="macro", zero_division=0)
    metrics["recall_macro"] = recall_score(true_dom, pred_dom, labels=label_ids, average="macro", zero_division=0)

    return metrics


# 8. Training Arguments & Trainer


In [12]:
print("\n[Step 8] Configuring training...")


class SoftLabelTrainer(Trainer):
    """
    Trainer whose loss is the cross-entropy between softmax(logits) and the soft
    "senti" target distribution. The model's built-in loss can't be used: it
    expects one class id per sample, not four scores.
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # compute_loss returns a per-batch mean, so the Trainer must divide it by
        # the gradient-accumulation steps itself.
        self.model_accepts_loss_kwargs = False

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        loss = F.cross_entropy(outputs.logits.float(), labels.float())
        return (loss, outputs) if return_outputs else loss


training_args = TrainingArguments(
    output_dir=config.OUTPUT_DIR,
    #overwrite_output_dir=True,

    # Training
    num_train_epochs=config.NUM_EPOCHS,
    per_device_train_batch_size=config.BATCH_SIZE,
    per_device_eval_batch_size=config.BATCH_SIZE * 2,  # Larger batch for eval
    gradient_accumulation_steps=config.GRADIENT_ACCUMULATION_STEPS,
    learning_rate=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY,
    # warmup_ratio=config.WARMUP_RATIO,
    fp16=config.FP16 and torch.cuda.is_available(),

    # Evaluation & Saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model=config.METRIC_FOR_BEST_MODEL,
    greater_is_better=config.GREATER_IS_BETTER,
    save_total_limit=2,              # Keep only last 2 checkpoints (saves disk)

    # Logging
    #logging_dir=config.LOGGING_DIR,
    logging_steps=100,
    report_to="none",                # Disable W&B/MLflow on Colab

    # Performance
    dataloader_num_workers=2,
    dataloader_pin_memory=True,

    # Reproducibility
    seed=42,
)

trainer = SoftLabelTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=config.EARLY_STOPPING_PATIENCE)
    ],
)


[Step 8] Configuring training...



# 9. Train


In [13]:
print("\n" + "=" * 60)
print("  STARTING TRAINING")
print("=" * 60)

train_result = trainer.train()

# Log training results
print("\n[Training Complete]")
print(f"  Training loss:       {train_result.training_loss:.4f}")
print(f"  Training runtime:    {train_result.metrics['train_runtime']:.1f}s")
print(f"  Samples/second:      {train_result.metrics['train_samples_per_second']:.1f}")


  STARTING TRAINING


Epoch,Training Loss,Validation Loss,Mae,Rmse,Mae Pos,Mae Neg,Mae Neu,Mae Amb,Accuracy,F1 Macro,Precision Macro,Recall Macro
1,0.941635,0.941385,0.243451,0.342849,0.260502,0.252886,0.321945,0.138472,0.617536,0.575672,0.579747,0.583668
2,0.898398,0.931341,0.232094,0.341247,0.247119,0.239473,0.306104,0.135678,0.624815,0.577426,0.583837,0.582970
3,0.862085,0.946806,0.224676,0.343281,0.246900,0.218440,0.294366,0.138999,0.626202,0.582501,0.582922,0.588118
4,0.838747,0.965447,0.225683,0.346804,0.250793,0.215493,0.291510,0.144936,0.619523,0.570814,0.575591,0.581952
5,0.801903,0.972073,0.223831,0.347847,0.240069,0.224609,0.292837,0.137809,0.619084,0.574037,0.575753,0.581885


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

: 


# 10. Evaluate on Validation Set


In [ ]:
print("\n[Step 10] Evaluating on validation set...")
val_metrics = trainer.evaluate()
print("\nValidation Metrics:")
for key, value in val_metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")


# 11. Evaluate on Test Set — Full Report


In [ ]:
print("\n[Step 11] Evaluating on test set...")
test_output = trainer.predict(test_dataset)
test_metrics = test_output.metrics

print("\nTest Metrics:")
for key, value in test_metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

# --- Scores & dominant sentiments ---
pred_scores = logits_to_scores(test_output.predictions)
true_scores = np.asarray(test_output.label_ids, dtype=np.float64)
true_dom = dominant_indices(true_scores)
pred_dom = dominant_indices(pred_scores)

# --- Per-sentiment score error ---
score_report_lines = [f"{'sentiment':<12}{'MAE':>8}{'RMSE':>8}"]
for i, label in enumerate(SENTIMENT_LABELS):
    mae = mean_absolute_error(true_scores[:, i], pred_scores[:, i])
    rmse = np.sqrt(mean_squared_error(true_scores[:, i], pred_scores[:, i]))
    score_report_lines.append(f"{label:<12}{mae:>8.4f}{rmse:>8.4f}")
score_report = "\n".join(score_report_lines)

print("\n" + "=" * 60)
print("  PER-SENTIMENT SCORE ERROR (0-1 scale)")
print("=" * 60)
print(score_report)

# --- Dominant-sentiment classification report ---
print("\n" + "=" * 60)
print("  DOMINANT-SENTIMENT CLASSIFICATION REPORT")
print("=" * 60)
report = classification_report(
    true_dom,
    pred_dom,
    labels=list(range(len(SENTIMENT_LABELS))),
    target_names=SENTIMENT_LABELS,
    zero_division=0,
)
print(report)

# Save the reports to file
with open(os.path.join(config.RESULTS_DIR, "classification_report.txt"), "w") as f:
    f.write("SentiVox — Test Set Report\n")
    f.write("=" * 60 + "\n")
    f.write("Per-Sentiment Score Error (0-1 scale)\n")
    f.write(score_report + "\n\n")
    f.write("Dominant-Sentiment Classification Report\n")
    f.write(report)


# 12. Confusion Matrix & Score Visualization


In [ ]:
print("\n[Step 12] Generating plots...")

def plot_sentiment_confusion_matrix(y_true_dom, y_pred_dom, save_path):
    """Confusion matrix of the dominant sentiment (true vs. predicted)."""
    cm = confusion_matrix(y_true_dom, y_pred_dom, labels=list(range(len(SENTIMENT_LABELS))))
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Purples",
        xticklabels=SENTIMENT_LABELS,
        yticklabels=SENTIMENT_LABELS,
        ax=ax,
    )
    ax.set_title("Dominant-Sentiment Confusion Matrix", fontsize=14, fontweight="bold")
    ax.set_xlabel("Predicted Sentiment")
    ax.set_ylabel("True Sentiment")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Sentiment confusion matrix saved → {save_path}")


def plot_score_scatter(y_true_scores, y_pred_scores, save_path):
    """True vs. predicted 0-1 score, one panel per sentiment."""
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    fig.suptitle("True vs. Predicted Sentiment Scores", fontsize=16, fontweight="bold")

    for i, (ax, label) in enumerate(zip(axes.flat, SENTIMENT_LABELS)):
        mae = mean_absolute_error(y_true_scores[:, i], y_pred_scores[:, i])
        ax.scatter(y_true_scores[:, i], y_pred_scores[:, i], s=6, alpha=0.3)
        ax.plot([0, 1], [0, 1], "r--", linewidth=1)
        ax.set_xlim(-0.05, 1.05)
        ax.set_ylim(-0.05, 1.05)
        ax.set_title(f"{label} (MAE {mae:.3f})", fontsize=11)
        ax.set_xlabel("True score")
        ax.set_ylabel("Predicted score")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Score scatter plot saved → {save_path}")


sentiment_cm_path = os.path.join(config.RESULTS_DIR, "confusion_matrix_sentiment.png")
score_scatter_path = os.path.join(config.RESULTS_DIR, "score_scatter.png")

plot_sentiment_confusion_matrix(true_dom, pred_dom, sentiment_cm_path)
plot_score_scatter(true_scores, pred_scores, score_scatter_path)


# 13. Save Metrics to JSON


In [ ]:
print("\n[Step 13] Saving metrics...")

all_metrics = {
    "language": "English",
    "base_model": config.MODEL_NAME,
    "dataset": DATASET_NAME,
    "labels": SENTIMENT_LABELS,
    "score_range": [0, 1],
    "scores_sum_to": 1,
    "loss": "soft-label cross-entropy over softmax(logits)",
    "training": {
        "epochs": config.NUM_EPOCHS,
        "batch_size": config.BATCH_SIZE,
        "effective_batch_size": config.BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": config.LEARNING_RATE,
        "training_loss": train_result.training_loss,
    },
    "validation_metrics": {
        k: round(v, 4) for k, v in val_metrics.items() if isinstance(v, float)
    },
    "test_metrics": {
        k: round(v, 4) for k, v in test_metrics.items() if isinstance(v, float)
    },
}

metrics_path = os.path.join(config.RESULTS_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"  Metrics saved → {metrics_path}")


# 14. Save Fine-Tuned Model (HuggingFace-deployable)


In [ ]:
print("\n[Step 14] Saving fine-tuned model...")

# Save model + tokenizer (HuggingFace format — ready for Hub push)
trainer.save_model(config.OUTPUT_DIR)
tokenizer.save_pretrained(config.OUTPUT_DIR)

# Save model card for HuggingFace Hub
model_card = """---
language:
  - en
  - hi
  - bn
  - ta
  - te
  - ml
  - kn
  - gu
  - mr
  - pa
  - or
  - as
  - ur
license: apache-2.0
tags:
  - sentiment-analysis
  - multilingual
  - muril
  - indian-languages
pipeline_tag: text-classification
---

# SentiVox — Multilingual Sentiment Analysis Model

Fine-tuned [google/muril-base-cased](https://huggingface.co/google/muril-base-cased)
to score text on four sentiment axes: positive, negative, ambiguous and neutral.

## Model Description

- **Base Model**: google/muril-base-cased (BERT-based, supports 17 Indian languages + English)
- **Task**: Sentiment distribution scoring — one 0-1 score per sentiment; the four scores sum to 1
- **Labels**: positive, negative, neutral, ambiguous
- **Dominant Sentiment**: the label with the highest score

The outputs are softmax probabilities, so a text that is both positive and negative
gets a share on each (e.g. positive 0.5, negative 0.4).
The model was trained on text cleaned by SentiVox's `clean_text` (scripts/tokenizer.py);
clean input text the same way before scoring it.

## Usage

```python
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="<your-hub-id>/sentivox-muril-sentiment",
    top_k=None,
    function_to_apply="softmax",
)
result = classifier(["i am so happy and grateful !"])[0]   # one list of 4 labels per input text
scores = {{r["label"]: round(r["score"], 4) for r in result}}
```

## Training Details

- **Dataset**: Sentiment-scored text (`{{"text", "senti": {{"pos", "neg", "neu", "amb"}}}}`, scores 0-1 that sum to 1)
- **Loss**: Soft-label cross-entropy between softmax(logits) and the `senti` scores
- **Epochs**: {epochs}
- **Batch Size**: {batch_size} (effective: {eff_batch})
- **Learning Rate**: {lr}
- **Optimizer**: AdamW with linear warmup
- **Hardware**: Google Colab (T4 GPU)

## Future Work

- Fine-tune on Hindi, Bengali, Tamil, and other Indian language datasets
- Add document-level context aggregation for longer texts
""".format(
    epochs=config.NUM_EPOCHS,
    batch_size=config.BATCH_SIZE,
    eff_batch=config.BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS,
    lr=config.LEARNING_RATE,
)

with open(os.path.join(config.OUTPUT_DIR, "README.md"), "w") as f:
    f.write(model_card)

print(f"  Model saved → {config.OUTPUT_DIR}")
print(f"  Model card saved → {os.path.join(config.OUTPUT_DIR, 'README.md')}")


# 15. Inference Utilities


In [ ]:
print("\n[Step 15] Setting up inference utilities...")


def load_sentivox_model(model_path: str = config.OUTPUT_DIR):
    """
    Load the fine-tuned SentiVox model and tokenizer.

    Args:
        model_path: Path to the saved model directory.

    Returns:
        Tuple of (model, tokenizer, device).
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)
    loaded_model = AutoModelForSequenceClassification.from_pretrained(model_path)
    loaded_model.to(device)
    loaded_model.eval()
    return loaded_model, loaded_tokenizer, device

from scripts.tokenizer import clean_text


def _scores_to_result(scores) -> dict:
    """Build the output dict for one 4-element score vector."""
    return {
        "sentiment": scores_to_sentiment(scores),
        "scores": {
            label: round(float(score), 4)
            for label, score in zip(SENTIMENT_LABELS, scores)
        },
    }


def predict_batch(
    texts: list[str],
    model_instance=None,
    tokenizer_instance=None,
    device_instance=None,
    batch_size: int = 32,
) -> list[dict]:
    """
    Run inference on a batch of texts.

    Args:
        texts:              List of input text strings.
        model_instance:     Loaded PyTorch model (uses global if None).
        tokenizer_instance: Loaded tokenizer (uses global if None).
        device_instance:    Torch device (uses global if None).
        batch_size:         Inference batch size.

    Returns:
        List of dictionaries, each with keys:
            - "sentiment": str — dominant sentiment (positive/negative/neutral/ambiguous)
            - "scores": dict[str, float] — 0-1 score per sentiment (the four sum to 1)
    """
    _model = model_instance or model
    _tokenizer = tokenizer_instance or tokenizer
    _device = device_instance or device

    results = []
    for i in range(0, len(texts), batch_size):
        # Clean the text the same way the training data was cleaned
        batch_texts = [clean_text(text) for text in texts[i : i + batch_size]]
        encoding = _tokenizer(
            batch_texts,
            max_length=config.MAX_LENGTH,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        encoding = {k: v.to(_device) for k, v in encoding.items()}

        with torch.no_grad():
            logits = _model(**encoding).logits.float().cpu().numpy()

        for scores in logits_to_scores(logits):
            results.append(_scores_to_result(scores))
    return results


def predict(
    text: str,
    model_instance=None,
    tokenizer_instance=None,
    device_instance=None,
) -> dict:
    """
    Run inference on a single text sample.

    Returns:
        Dictionary with keys:
            - "sentiment": str — dominant sentiment (positive/negative/neutral/ambiguous)
            - "scores": dict[str, float] — 0-1 score per sentiment (the four sum to 1)
    """
    return predict_batch([text], model_instance, tokenizer_instance, device_instance)[0]


def predict_document(
    paragraphs: list[str],
    model_instance=None,
    tokenizer_instance=None,
    device_instance=None,
) -> dict:
    """
    Document-level sentiment analysis.

    Scores each paragraph/sentence, then averages the scores across segments
    to determine the overall document sentiment.

    Args:
        paragraphs: List of text segments (paragraphs or sentences).

    Returns:
        Dictionary with:
            - "overall_sentiment": str
            - "overall_scores": dict[str, float] — mean 0-1 score per sentiment (sums to 1)
            - "segment_predictions": list[dict] — per-segment predictions
    """
    segment_preds = predict_batch(
        paragraphs, model_instance, tokenizer_instance, device_instance,
    )

    mean_scores = np.mean(
        [[pred["scores"][label] for label in SENTIMENT_LABELS] for pred in segment_preds],
        axis=0,
    )
    overall = _scores_to_result(mean_scores)

    return {
        "overall_sentiment": overall["sentiment"],
        "overall_scores": overall["scores"],
        "segment_predictions": segment_preds,
    }


# 16. Demo Predictions


In [ ]:
print("\n" + "=" * 60)
print("  DEMO PREDICTIONS")
print("=" * 60)

demo_texts = [
    "I am so happy and grateful for everything you have done!",
    "This is absolutely terrible and disgusting behavior.",
    "I'm not sure what to think about this situation.",
    "The weather is cloudy today.",
    "I love this but I'm also terrified of what comes next.",
    "He is not bad",
    "He is not good"
]

for text in demo_texts:
    result = predict(text)
    print(f"\n  Text: \"{text}\"")
    print(f"  Sentiment: {result['sentiment']}")
    print(f"  Scores:    {result['scores']}")

# Document-level demo
print("\n" + "-" * 40)
print("  Document-Level Analysis Demo")
print("-" * 40)

doc_paragraphs = [
    "The product arrived on time and was well packaged.",
    "However, the quality was much lower than expected.",
    "I contacted support and they were incredibly helpful and resolved it quickly.",
]

doc_result = predict_document(doc_paragraphs)
print(f"\n  Overall Sentiment: {doc_result['overall_sentiment']}")
print(f"  Overall Scores:    {doc_result['overall_scores']}")
for i, seg in enumerate(doc_result["segment_predictions"]):
    print(f"  Segment {i + 1}: {seg['sentiment']} — {seg['scores']}")


# 17. Summary


In [ ]:
print("\n" + "=" * 60)
print("  TRAINING PIPELINE COMPLETE")
print("=" * 60)
print(f"""
  Model:          {config.MODEL_NAME}
  Saved to:       {config.OUTPUT_DIR}
  Metrics:        {metrics_path}
  Report:         {os.path.join(config.RESULTS_DIR, 'classification_report.txt')}

  To push to HuggingFace Hub:
    from huggingface_hub import login, upload_folder

    # Login with your Hugging Face credentials
    login()

    # Push your model files
    upload_folder(
        folder_path="{config.OUTPUT_DIR}",
        repo_id="<your-user-name>/sentivox-muril-sentiment",
        repo_type="model"
    )

  To load for inference:
    model, tokenizer, device = load_sentivox_model("{config.OUTPUT_DIR}")
    result = predict("Your text here", model, tokenizer, device)
""")

In [ ]:
! du -h models/sentivox-muril-sentiment/model.safetensors

In [ ]:
from huggingface_hub import login, upload_folder

# (optional) Login with your Hugging Face credentials
login()

# Push your model files
upload_folder(
    folder_path="./models/sentivox-muril-sentiment",
    repo_id="saksham-joshi/sentivox-muril-sentiment",
    repo_type="model"
)